# Fly trajectory pattern analysis — Exp2 (w1118 control vs dehydrated)

**Workflow (run top to bottom):**

1. **Imports / setup** — load libraries and configure paths, palette, helpers.
2. **w1118 vs dehydrated** — pool `sd_09_08` + `sd_09_09`, label each video from `Sign-up list (1).pdf` (control vs dehydrated), then run the Step 0–style pattern analysis on that two-group comparison only.

Outputs go to `pattern_distance_output/w1118_dehydration_step0/<N>mm/` (one folder per segment length).


In [3]:
import sys
from pathlib import Path
from typing import Optional

# Expect this repo's `.venv` (requirements.txt). Wrong kernel → missing seaborn/sklearn, etc.
def _project_venv_python(start: Path) -> Optional[Path]:
    for d in [start, *start.parents][:12]:
        cand = d / ".venv" / "bin" / "python"
        if cand.is_file():
            return cand.resolve()
    return None


_root = Path.cwd()
_venv_py = _project_venv_python(_root)
if _venv_py is not None and Path(sys.executable).resolve() != _venv_py:
    raise RuntimeError(
        "Wrong Python interpreter for this notebook.\n"
        f"  Currently using: {sys.executable}\n"
        f"  Use instead:     {_venv_py}\n\n"
        "In Cursor: kernel picker (top-right) → **Python (.venv Experiment1)**. "
        "Or Command Palette → **Notebook: Select Notebook Kernel**.\n"
        "If that kernel is missing, in Terminal:\n"
        f"  cd {_venv_py.parent.parent.parent} && python3 -m venv .venv && "
        ".venv/bin/pip install -r requirements.txt && .venv/bin/python -m ipykernel install --user "
        "--name=experiment1-pattern --display-name='Python (.venv Experiment1)'"
    )

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from typing import Dict, Tuple

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings("ignore")

sns.set_context("talk")
sns.set_style("whitegrid")

# Exp2 reads raw detection CSVs from sd_09_08/ and sd_09_09/ directly; no merged_detection_all.csv needed.
DEFAULT_MM_PER_PX_X = 0.04
DEFAULT_MM_PER_PX_Y = 0.05
MM_PER_PX_BY_RUN: Dict[Tuple[str, int], Tuple[float, float]] = {}

PLOT_DIR = Path("pattern_distance_output")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

from pattern_export_stats import export_pattern_feature_stats_csv, export_pca_specification_csv

# SKIP_PATTERN_FIGURES — True = skip regenerating pattern PNGs only; clustering unchanged.
SKIP_PATTERN_FIGURES = False
# If True: only write cluster_naming_order.csv (skip segment_measures & other exports). Clustering time unchanged.
ONLY_CLUSTER_NAMING_ORDER_CSV = False


def save_fig(name: str, fig=None):
    if fig is None:
        fig = plt.gcf()
    out = PLOT_DIR / name
    fig.savefig(out, dpi=200, bbox_inches="tight")
    return out

FEATURE_COLS = ["straightness", "tortuosity", "path_per_40", "mean_abs_turn_deg", "n_turns"]
# Turn threshold (deg): direction change above this counts as a turn. Literature: 10-15° conservative; gait/locomotion often 22-45°.
TURN_THRESHOLD_DEG = 15
def turn_stats_from_xy(x, y):
    """From (x, y) points, return (mean_abs_turn_deg, n_turns). n_turns = count of direction changes > TURN_THRESHOLD_DEG."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    if len(x) < 3:
        return np.nan, 0
    dx = np.diff(x)
    dy = np.diff(y)
    head = np.arctan2(dy, dx)
    turn = np.diff(head)
    turn = np.where(turn > np.pi, turn - 2 * np.pi, np.where(turn < -np.pi, turn + 2 * np.pi, turn))
    deg = np.degrees(np.abs(turn))
    mean_deg = np.nanmean(deg) if len(deg) else np.nan
    n_turns = np.sum(deg > TURN_THRESHOLD_DEG) if len(deg) else 0
    return mean_deg, int(n_turns)

## Pooled `sd_09_08` + `sd_09_09`: w1118 (control) vs dehydrated w1118

The **code cell below** reads the raw `detection_output_*_filtered.csv` files from `sd_09_08/` and `sd_09_09/` directly (no `merged_detection_all.csv` required), labels each video from **Sign-up list (1).pdf**, recomputes kinematics, and writes outputs (CSVs + PNGs) to `pattern_distance_output/w1118_dehydration_step0/<N>mm/`.

Run the **imports** cell first (`DEFAULT_MM_PER_PX_*`, `save_fig`, `FEATURE_COLS`, `turn_stats_from_xy`, `SKIP_PATTERN_FIGURES`, `ONLY_CLUSTER_NAMING_ORDER_CSV`).

- **SD 9-9-25** (`sd_09_09`): videos **1, 3, 5** → w1118 (control on media); **2, 4, 6** → dehydrated w1118.
- **SD 9-8-25** (`sd_09_08`): videos **1–3** → w1118 (control on media); **4–7** → dehydrated w1118.

`detection_run_id` = video index (`N` in the filename).


In [4]:
# Helper function: previously lived in pattern_dehydration_step0.py — now inlined here
# so the whole pipeline is in this notebook (no external module to keep in sync).
# Uses np, pd, plt, sns, StandardScaler, KMeans, PCA, silhouette_score,
# export_pattern_feature_stats_csv, export_pca_specification_csv from the imports cell.

from typing import Any, Callable, Iterable, List, Optional

def run_dehydration_step0_style(
    df_sub: pd.DataFrame,
    video_cols: List[str],
    feature_cols: List[str],
    turn_stats_from_xy: Callable[..., tuple],
    out_base: Path,
    *,
    distances_mm: Optional[Iterable[int]] = None,
    min_path_mm: float = 5.0,
    hue_values: tuple[str, str] = ("young", "old"),
    hue_display_names: Optional[tuple[str, str]] = None,
    hue_labels: Optional[tuple[str, str]] = None,
    title_prefix: str = "W1118 dehydration — pooled sd_09_08 + sd_09_09",
    skip_figures: bool = False,
    only_cluster_naming_order_csv: bool = False,
    **kwargs: Any,
) -> pd.DataFrame:
    """
    df_sub must include: frame, time_s, x, y, cum_dist_mm, total_dist_mm, step_dist_mm, speed_mm_s,
    mm_per_px_x, mm_per_px_y, age_group (only the two strings in hue_values, for plot colours),
    category (any per-row label), plus video_cols.

    ``hue_display_names``: legend / count labels (default ``w1118``, ``Dehydrated w1118``).
    ``hue_labels``: ignored; kept for compatibility.

    Unknown keyword arguments are ignored (avoids ``TypeError`` after notebook/API tweaks).
    """
    if kwargs:
        pass  # e.g. stale kw from an older notebook cell

    distances_mm = list(distances_mm or [5, 10, 15, 20, 30, 40])
    out_base = Path(out_base)
    out_base.mkdir(parents=True, exist_ok=True)

    df_work = df_sub.dropna(subset=["cum_dist_mm", "total_dist_mm"]).copy()
    df_work = df_work[df_work["total_dist_mm"] > 0].copy()
    if len(df_work) == 0:
        raise ValueError("run_dehydration_step0_style: no rows after cum_dist/total_dist filter")

    summary_rows: list[dict] = []

    for seg_mm in distances_mm:
        out_dir = out_base / f"{seg_mm}mm"
        out_dir.mkdir(parents=True, exist_ok=True)
        SEGMENT_LENGTH_MM = float(seg_mm)

        df_seg = df_work.copy()
        df_seg["seg_40"] = (df_seg["cum_dist_mm"] // SEGMENT_LENGTH_MM).astype(int) + 1
        n_segs_per_video = (
            df_seg.groupby(video_cols)["total_dist_mm"].transform("first") / SEGMENT_LENGTH_MM
        ).apply(np.ceil).astype(int)
        df_seg["seg_40"] = df_seg["seg_40"].clip(upper=n_segs_per_video)

        seg = (
            df_seg.groupby(video_cols + ["category", "age_group", "seg_40"], as_index=False)
            .agg(
                n_rows=("frame", "size"),
                t_start_s=("time_s", "min"),
                t_end_s=("time_s", "max"),
                seg_path_mm=("step_dist_mm", "sum"),
                speed_mean_mm_s=("speed_mm_s", "mean"),
                x_start=("x", "first"),
                y_start=("y", "first"),
                x_end=("x", "last"),
                y_end=("y", "last"),
            )
        )
        seg["seg_dur_s"] = seg["t_end_s"] - seg["t_start_s"]
        seg["net_dx_px"] = seg["x_end"] - seg["x_start"]
        seg["net_dy_px"] = seg["y_end"] - seg["y_start"]
        seg["net_disp_px"] = np.sqrt(seg["net_dx_px"] ** 2 + seg["net_dy_px"] ** 2)
        cal = df_seg.groupby(video_cols, as_index=False).agg(
            mm_per_px_x=("mm_per_px_x", "first"),
            mm_per_px_y=("mm_per_px_y", "first"),
        )
        seg = seg.merge(cal, on=video_cols, how="left")
        seg["net_disp_mm"] = np.sqrt(
            (seg["net_dx_px"] * seg["mm_per_px_x"]) ** 2 + (seg["net_dy_px"] * seg["mm_per_px_y"]) ** 2
        )
        seg["straightness"] = seg["net_disp_mm"] / seg["seg_path_mm"].replace({0: np.nan})
        seg = seg[seg["seg_path_mm"] >= min_path_mm].copy()
        seg["tortuosity"] = seg["seg_path_mm"] / seg["net_disp_mm"].replace(0, np.nan).clip(lower=1e-6)
        seg["path_per_40"] = seg["seg_path_mm"] / SEGMENT_LENGTH_MM

        mean_turns, n_turns_list = [], []
        for _, r in seg.iterrows():
            sub = df_seg[
                (df_seg[video_cols[0]] == r[video_cols[0]])
                & (df_seg[video_cols[1]] == r[video_cols[1]])
                & (df_seg["seg_40"] == r["seg_40"])
            ].sort_values(["frame", "time_s"])
            ma, nt = turn_stats_from_xy(sub["x"].values, sub["y"].values)
            mean_turns.append(ma)
            n_turns_list.append(nt)
        seg["mean_abs_turn_deg"] = mean_turns
        seg["n_turns"] = n_turns_list
        seg = seg.dropna(subset=["straightness", "tortuosity", "path_per_40"]).copy()
        seg["mean_abs_turn_deg"] = seg["mean_abs_turn_deg"].fillna(0)

        if len(seg) < 10:
            print(f"  {title_prefix} {seg_mm}mm: too few segments ({len(seg)}), skip")
            continue

        X = seg[feature_cols].copy()
        _feat_scaler = StandardScaler()
        X_scaled = _feat_scaler.fit_transform(X)
        K_RANGE = range(2, max(3, min(11, len(seg) // 5)))
        silhouettes, inertias = [], []
        for k in K_RANGE:
            km_trial = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
            silhouettes.append(silhouette_score(X_scaled, km_trial.predict(X_scaled)))
            inertias.append(km_trial.inertia_)
        best_k = list(K_RANGE)[np.argmax(silhouettes)] if silhouettes else 2
        n_clusters = best_k
        km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        seg["pattern"] = km.fit_predict(X_scaled)
        seg["pattern"] = seg["pattern"].astype(int)
        seg_clean = seg.copy()

        means = seg_clean.groupby("pattern")[feature_cols].mean()
        patterns = sorted(means.index.tolist())
        assigned: set[int] = set()

        def assign_best(lbl: str, feature: str, order: str = "max") -> Optional[int]:
            remaining = [p for p in patterns if p not in assigned]
            if not remaining:
                return None
            pat = (
                means.loc[remaining, feature].idxmax()
                if order == "max"
                else means.loc[remaining, feature].idxmin()
            )
            assigned.add(pat)
            return pat

        label_specs = [
            ("zig-zag", "n_turns", "max"),
            ("straight", "straightness", "max"),
            ("meandering", "path_per_40", "max"),
            ("curved", "mean_abs_turn_deg", "max"),
            ("winding", "tortuosity", "max"),
            ("direct", "straightness", "max"),
            ("exploratory", "path_per_40", "max"),
        ]
        pattern_labels: dict[int, str] = {}
        for lab, feat, ord_ in label_specs[:n_clusters]:
            p = assign_best(lab, feat, ord_)
            if p is not None:
                pattern_labels[p] = lab
        for p in patterns:
            if p not in pattern_labels:
                pattern_labels[p] = "moderate"
        seg_clean["pattern_label"] = seg_clean["pattern"].map(pattern_labels)
        PATTERN_GROUP = {
            "straight": "Straight",
            "direct": "Direct",
            "meandering": "Meandering",
            "zig-zag": "Zig-zag / reversing",
            "curved": "Curved / turning",
            "winding": "Winding",
            "exploratory": "Exploratory",
            "moderate": "Other / moderate",
        }
        seg_clean["pattern_group"] = seg_clean["pattern_label"].map(PATTERN_GROUP)

        distinct_names = sorted(set(seg_clean["pattern_label"].astype(str)))
        best_sil = float(max(silhouettes)) if silhouettes else float("nan")
        summary_rows.append(
            {
                "segment_length_mm": seg_mm,
                "n_segments": len(seg_clean),
                "n_patterns_k": int(n_clusters),
                "n_distinct_labels": len(distinct_names),
                "best_silhouette": best_sil,
                "labels": "; ".join(distinct_names),
            }
        )
        print(
            f"  {title_prefix} {seg_mm}mm: n_segments={len(seg_clean):,}  ->  k={n_clusters}  "
            f"silhouette={best_sil:.4f}"
        )

        export_pattern_feature_stats_csv(
            seg_clean,
            feature_cols,
            out_dir,
            segment_length_mm=seg_mm,
            step_name="w1118_dehydration_step0",
            only_cluster_naming_order_csv=only_cluster_naming_order_csv,
        )
        _pca = PCA(n_components=2, random_state=42)
        pc_xy = _pca.fit_transform(X_scaled)
        export_pca_specification_csv(
            _feat_scaler,
            _pca,
            feature_cols,
            out_dir,
            analysis_id="w1118_dehydration_step0",
            segment_length_mm=seg_mm,
            n_segments=len(seg_clean),
            k_patterns=int(n_clusters),
        )
        if skip_figures:
            continue

        h0, h1 = hue_values
        if hue_display_names is not None:
            name0, name1 = hue_display_names
        else:
            name0, name1 = ("w1118", "Dehydrated w1118")
        pal = {h0: "steelblue", h1: "coral"}

        # Plot 1
        fig, ax = plt.subplots(1, 2, figsize=(10, 4))
        ax[0].plot(list(K_RANGE), silhouettes, "o-")
        ax[0].axvline(n_clusters, color="green", linestyle="--", alpha=0.8, label=f"chosen k={n_clusters}")
        ax[0].set_xlabel("k (number of patterns)")
        ax[0].set_ylabel("Silhouette score")
        ax[0].legend()
        ax[1].plot(list(K_RANGE), inertias, "o-")
        ax[1].axvline(n_clusters, color="green", linestyle="--", alpha=0.8)
        ax[1].set_xlabel("k")
        ax[1].set_ylabel("Inertia")
        plt.suptitle(f"{title_prefix} — {seg_mm} mm segments", y=1.02)
        plt.tight_layout()
        fig.savefig(out_dir / "step0_silhouette_k.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

        # Plot 2
        fig, ax = plt.subplots(figsize=(9, 4))
        seg_clean["pattern_label"].value_counts().sort_index().plot(
            kind="bar", ax=ax, color="steelblue", edgecolor="gray"
        )
        ax.set_ylabel("Segment count")
        ax.set_xlabel("Pattern label")
        ax.set_title(f"{title_prefix}: segments per pattern (k={n_clusters}, {seg_mm} mm)")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        fig.savefig(out_dir / "step0_pattern_counts.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

        # Plot 3
        seg_clean["PC1"] = pc_xy[:, 0]
        seg_clean["PC2"] = pc_xy[:, 1]
        fig, ax = plt.subplots(figsize=(10, 6.5))
        sns.scatterplot(data=seg_clean, x="PC1", y="PC2", hue="pattern_label", alpha=0.25, s=8, ax=ax)
        ax.set_title(f"{title_prefix} — PCA {seg_mm} mm (k={n_clusters})")
        ax.legend(
            bbox_to_anchor=(0.5, -0.14),
            loc="upper center",
            ncol=3,
            fontsize=8,
            title="Pattern",
            title_fontsize=9,
            frameon=True,
            columnspacing=0.9,
            handletextpad=0.4,
        )
        fig.tight_layout()
        fig.subplots_adjust(bottom=0.24)
        fig.savefig(out_dir / "step0_pca_patterns.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

        # Plot 4
        groups_sorted = seg_clean["pattern_group"].dropna().unique()
        groups_sorted = sorted(groups_sorted, key=lambda g: -(seg_clean["pattern_group"] == g).sum())
        n_groups = len(groups_sorted)
        n_cols = min(3, max(1, n_groups))
        n_rows = int(np.ceil(n_groups / n_cols)) if n_groups else 1
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
        axes = np.atleast_2d(axes)

        for idx_g, grp in enumerate(groups_sorted):
            ax = axes.flat[idx_g]
            subset = seg_clean[seg_clean["pattern_group"] == grp]
            for hue_val in hue_values:
                sub = subset[subset["age_group"] == hue_val]
                if len(sub) == 0:
                    continue
                for _, r in sub.iterrows():
                    pts = df_seg[
                        (df_seg[video_cols[0]] == r[video_cols[0]])
                        & (df_seg[video_cols[1]] == r[video_cols[1]])
                        & (df_seg["seg_40"] == r["seg_40"])
                    ].sort_values(["frame", "time_s"])
                    x = pts["x"].values.astype(float) - pts["x"].iloc[0]
                    y = pts["y"].values.astype(float) - pts["y"].iloc[0]
                    ax.plot(
                        x,
                        y,
                        color=pal[hue_val],
                        alpha=0.20,
                        linewidth=0.7,
                    )
            n_a = (subset["age_group"] == h0).sum()
            n_b = (subset["age_group"] == h1).sum()
            tit = f"{grp}\n{name0} : {n_a}    {name1} : {n_b}."
            if "Curved" in grp:
                tit += f"\navg turn {subset['mean_abs_turn_deg'].mean():.0f}°"
            elif "Zig-zag" in grp:
                tit += f"\n{subset['n_turns'].mean():.1f} turns/seg"
            ax.set_title(tit, fontsize=9)
            ax.set_xlabel("x (px from start)", fontsize=8)
            ax.set_ylabel("y (px from start)", fontsize=8)
            ax.axis("equal")

        for j in range(n_groups, axes.size):
            axes.flat[j].set_visible(False)

        plt.suptitle(
            f"{title_prefix} — movement ({seg_mm} mm)  "
            f"steelblue = {name0} :    coral = {name1} :",
            y=1.02,
            fontsize=11,
        )
        plt.tight_layout()
        fig.savefig(out_dir / "step0_movements_per_pattern.png", dpi=200, bbox_inches="tight")
        plt.close(fig)

        fig_b1, ax_b1 = plt.subplots(figsize=(10, 5))
        sns.boxplot(
            data=seg_clean,
            x="pattern_group",
            y="mean_abs_turn_deg",
            hue="age_group",
            palette=pal,
            ax=ax_b1,
        )
        ax_b1.set_title("Avg turn angle per pattern")
        ax_b1.set_xticklabels(ax_b1.get_xticklabels(), rotation=20, ha="right", fontsize=8)
        _h1, _lab1 = ax_b1.get_legend_handles_labels()
        _lab1_new = [name0 if str(x) == str(h0) else name1 for x in _lab1]
        ax_b1.legend(_h1, _lab1_new, bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
        fig_b1.tight_layout(rect=[0, 0, 0.78, 1])
        fig_b1.savefig(out_dir / "step0_movements_avg_turn_per_pattern.png", dpi=200, bbox_inches="tight")
        plt.close(fig_b1)

        fig_b2, ax_b2 = plt.subplots(figsize=(10, 5))
        sns.boxplot(
            data=seg_clean,
            x="pattern_group",
            y="n_turns",
            hue="age_group",
            palette=pal,
            ax=ax_b2,
        )
        ax_b2.set_title("Number of turns per pattern")
        ax_b2.set_xticklabels(ax_b2.get_xticklabels(), rotation=20, ha="right", fontsize=8)
        _h2, _lab2 = ax_b2.get_legend_handles_labels()
        _lab2_new = [name0 if str(x) == str(h0) else name1 for x in _lab2]
        ax_b2.legend(_h2, _lab2_new, bbox_to_anchor=(1.02, 1), loc="upper left", borderaxespad=0)
        fig_b2.tight_layout(rect=[0, 0, 0.78, 1])
        fig_b2.savefig(out_dir / "step0_movements_n_turns_per_pattern.png", dpi=200, bbox_inches="tight")
        plt.close(fig_b2)

        fig_s, ax_s = plt.subplots(figsize=(10, 6.5))
        sns.scatterplot(data=seg_clean, x="mean_abs_turn_deg", y="n_turns", hue="pattern_group", alpha=0.35, s=10, ax=ax_s)
        ax_s.set_title("Turn angle vs n_turns")
        ax_s.legend(
            bbox_to_anchor=(0.5, -0.14),
            loc="upper center",
            ncol=3,
            fontsize=8,
            title="Pattern",
            title_fontsize=9,
            frameon=True,
            columnspacing=0.9,
            handletextpad=0.4,
        )
        fig_s.tight_layout()
        fig_s.subplots_adjust(bottom=0.24)
        fig_s.savefig(out_dir / "step0_movements_turn_angle_vs_n_turns.png", dpi=200, bbox_inches="tight")
        plt.close(fig_s)

        # Plot 5
        n_cols5 = min(3, n_groups)
        n_rows5 = int(np.ceil(n_groups / n_cols5))
        fig5, axes5 = plt.subplots(n_rows5, n_cols5, figsize=(6 * n_cols5, 5 * n_rows5))
        axes5 = np.atleast_2d(axes5)

        for idx_g, grp in enumerate(groups_sorted):
            ax5 = axes5.flat[idx_g]
            subset = seg_clean[seg_clean["pattern_group"] == grp]
            for hue_val in hue_values:
                sub = subset[subset["age_group"] == hue_val]
                if len(sub) == 0:
                    continue
                for _, r in sub.iterrows():
                    pts = df_seg[
                        (df_seg[video_cols[0]] == r[video_cols[0]])
                        & (df_seg[video_cols[1]] == r[video_cols[1]])
                        & (df_seg["seg_40"] == r["seg_40"])
                    ].sort_values(["frame", "time_s"])
                    ax5.plot(
                        pts["x"].values,
                        pts["y"].values,
                        color=pal[hue_val],
                        alpha=0.30,
                        linewidth=0.8,
                    )
            n_a = (subset["age_group"] == h0).sum()
            n_b = (subset["age_group"] == h1).sum()
            ax5.set_title(f"{grp}\n{name0} : {n_a}    {name1} : {n_b}.", fontsize=9)
            ax5.set_xlabel("x (px)", fontsize=8)
            ax5.set_ylabel("y (px)", fontsize=8)
            ax5.invert_yaxis()
            ax5.set_aspect("equal", adjustable="datalim")

        for j in range(n_groups, axes5.size):
            axes5.flat[j].set_visible(False)

        plt.suptitle(
            f"{title_prefix} — vial positions ({seg_mm} mm)  "
            f"steelblue = {name0} :    coral = {name1} :",
            y=1.02,
            fontsize=11,
        )
        plt.tight_layout()
        fig5.savefig(out_dir / "step0_actual_positions.png", dpi=200, bbox_inches="tight")
        plt.close(fig5)

    sum_df = pd.DataFrame(summary_rows)
    sum_path = out_base / "step0_pattern_summary.csv"
    sum_df.to_csv(sum_path, index=False)
    print(f"Summary: {sum_path.resolve()}")
    return sum_df


In [ ]:
# --- w1118 vs dehydrated: read individual detection_output_*.csv (sd_09_08, sd_09_09) ---
# No merged_detection_all.csv. Expect folders next to this notebook (same layout as merge_detection_csvs.py):
#   .../sd_09_08/detection_output_<N>.csv or detection_output_<N>_filtered.csv
#   .../sd_09_09/detection_output_<N>.csv or detection_output_<N>_filtered.csv
# Requires the **imports** cell: np, pd, sns, plt, Path, Optional, DEFAULT_MM_PER_PX_*,
# MM_PER_PX_BY_RUN, PLOT_DIR, save_fig.

import re

from IPython.display import display, Markdown

FILENAME_RE = re.compile(
    r"detection_output_(\d+)(?:_filtered)?\.csv$", re.IGNORECASE
)

# Folders sd_09_08 / sd_09_09 live here (change if your raw CSVs are elsewhere).
DEHYDRATION_DATA_BASE = Path.cwd()

PDF_NOTES = r"""
### Sign-up list.pdf — which `detection_output_N.csv` is which

**SD 9-9-25** (folder `sd_09_09`, experiment id `sd_09_09`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1, 3, 5: w1118** · **Videos 2, 4, 6: dehydrated w1118**

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` | **w1118** (on media) |
| `detection_output_2.csv` | **dehydrated w1118** |
| `detection_output_3.csv` | **w1118** (on media) |
| `detection_output_4.csv` | **dehydrated w1118** |
| `detection_output_5.csv` | **w1118** (on media) |
| `detection_output_6.csv` | **dehydrated w1118** |

---

**SD 9-8-25** (folder `sd_09_08`, experiment id `sd_09_08`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1–3: On media** · **Videos 4–7: Dehydrated**  
(We plot these as **w1118 / on-media** vs **w1118 dehydrated** to match the 9-9 cohort design.)

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` … `detection_output_3.csv` | **On media** → pooled as **w1118 (on media)** |
| `detection_output_4.csv` … `detection_output_7.csv` | **Dehydrated** → pooled as **w1118 dehydrated** |

**N** is the video index (same as **`detection_run_id`**). Files may be named `detection_output_N.csv` or `detection_output_N_filtered.csv`.
"""
display(Markdown(PDF_NOTES))

POOL_EXPS = ("sd_09_08", "sd_09_09")


def signup_w1118_condition(exp: str, detection_run_id: int) -> Optional[str]:
    e, r = str(exp), int(detection_run_id)
    if e == "sd_09_09":
        if r in (1, 3, 5):
            return "w1118 (on media)"
        if r in (2, 4, 6):
            return "w1118 dehydrated"
        return None
    if e == "sd_09_08":
        if r in (1, 2, 3):
            return "w1118 (on media)"
        if r in (4, 5, 6, 7):
            return "w1118 dehydrated"
        return None
    return None


rows = []
for exp in POOL_EXPS:
    n_vid = 7 if exp == "sd_09_08" else 6
    for vid in range(1, n_vid + 1):
        rows.append((exp, vid, signup_w1118_condition(exp, vid)))
signup_tbl = pd.DataFrame(
    rows, columns=["experiment_id", "detection_run_id (video #)", "condition (from PDF)"]
)
display(signup_tbl)

pdf_csv_map = pd.DataFrame(
    [
        ("sd_09_09", 1, "detection_output_1.csv", "w1118 (PDF: video 1)"),
        ("sd_09_09", 2, "detection_output_2.csv", "dehydrated w1118 (PDF: video 2)"),
        ("sd_09_09", 3, "detection_output_3.csv", "w1118 (PDF: video 3)"),
        ("sd_09_09", 4, "detection_output_4.csv", "dehydrated w1118 (PDF: video 4)"),
        ("sd_09_09", 5, "detection_output_5.csv", "w1118 (PDF: video 5)"),
        ("sd_09_09", 6, "detection_output_6.csv", "dehydrated w1118 (PDF: video 6)"),
        ("sd_09_08", 1, "detection_output_1.csv", "on media (PDF) → w1118 (on media)"),
        ("sd_09_08", 2, "detection_output_2.csv", "on media (PDF) → w1118 (on media)"),
        ("sd_09_08", 3, "detection_output_3.csv", "on media (PDF) → w1118 (on media)"),
        ("sd_09_08", 4, "detection_output_4.csv", "Dehydrated (PDF) → w1118 dehydrated"),
        ("sd_09_08", 5, "detection_output_5.csv", "Dehydrated (PDF) → w1118 dehydrated"),
        ("sd_09_08", 6, "detection_output_6.csv", "Dehydrated (PDF) → w1118 dehydrated"),
        ("sd_09_08", 7, "detection_output_7.csv", "Dehydrated (PDF) → w1118 dehydrated"),
    ],
    columns=["experiment_id", "N (video #)", "CSV filename", "PDF → plot group"],
)
print("PDF → which CSV (same rules as `signup_w1118_condition`):")
display(pdf_csv_map)


def load_raw_detection_pool(base: Path, experiment_ids: tuple[str, ...]) -> pd.DataFrame:
    """Concatenate raw detection CSVs; add experiment_id + detection_run_id from path."""
    parts: list[pd.DataFrame] = []
    for exp in experiment_ids:
        exp_dir = base / exp
        if not exp_dir.is_dir():
            print(f"Skip (folder missing): {exp_dir}")
            continue
        for csv_path in sorted(exp_dir.glob("detection_output_*.csv")):
            m = FILENAME_RE.search(csv_path.name)
            if not m:
                continue
            run_id = int(m.group(1))
            chunk = pd.read_csv(csv_path, low_memory=False)
            req = {"frame", "time_s", "x", "y"}
            miss = req - set(chunk.columns)
            if miss:
                raise ValueError(f"{csv_path}: missing columns {sorted(miss)}")
            chunk = chunk.copy()
            chunk["experiment_id"] = exp
            chunk["detection_run_id"] = run_id
            parts.append(chunk)
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)


print(f"Reading individual CSVs under {DEHYDRATION_DATA_BASE.resolve()}")
sub = load_raw_detection_pool(DEHYDRATION_DATA_BASE, POOL_EXPS)

if sub.empty:
    print(
        "No matching detection CSVs. Expected `detection_output_<N>.csv` or "
        "`detection_output_<N>_filtered.csv` under:\n"
        f"  {DEHYDRATION_DATA_BASE / 'sd_09_08'}/\n"
        f"  {DEHYDRATION_DATA_BASE / 'sd_09_09'}/\n"
        "Or set DEHYDRATION_DATA_BASE to the parent folder that contains those two subfolders."
    )
else:
    video_cols = ["experiment_id", "detection_run_id"]
    vkeys = sub[video_cols].drop_duplicates().reset_index(drop=True)
    mmx, mmy = [], []
    for _, r in vkeys.iterrows():
        key = (str(r["experiment_id"]), int(r["detection_run_id"]))
        tx, ty = MM_PER_PX_BY_RUN.get(key, (DEFAULT_MM_PER_PX_X, DEFAULT_MM_PER_PX_Y))
        mmx.append(tx)
        mmy.append(ty)
    vkeys["mm_per_px_x"] = mmx
    vkeys["mm_per_px_y"] = mmy
    sub = sub.merge(vkeys, on=video_cols, how="left")
    sub = sub.sort_values(video_cols + ["frame", "time_s"]).reset_index(drop=True)

    g = sub.groupby(video_cols, sort=False)
    sub["dt_s"] = g["time_s"].diff()
    sub["dx_px"] = g["x"].diff()
    sub["dy_px"] = g["y"].diff()
    sub.loc[sub["dt_s"] == 0, "dt_s"] = np.nan
    dx_mm = sub["dx_px"] * sub["mm_per_px_x"]
    dy_mm = sub["dy_px"] * sub["mm_per_px_y"]
    sub["step_dist_mm"] = np.sqrt(dx_mm**2 + dy_mm**2).fillna(0.0)
    sub["speed_mm_s"] = sub["step_dist_mm"] / sub["dt_s"]
    sub["cum_dist_mm"] = g["step_dist_mm"].cumsum()
    total = (
        sub.groupby(video_cols, as_index=False)["cum_dist_mm"]
        .max()
        .rename(columns={"cum_dist_mm": "total_dist_mm"})
    )
    sub = sub.merge(total, on=video_cols, how="left")
    print(f"Rows: {len(sub):,}  videos: {sub[video_cols].drop_duplicates().shape[0]}")

    sub["condition"] = [
        signup_w1118_condition(e, r) for e, r in zip(sub["experiment_id"], sub["detection_run_id"])
    ]
    ok = sub[sub["condition"].notna()].copy()
    miss = sub[sub["condition"].isna()][["experiment_id", "detection_run_id"]].drop_duplicates()
    if len(miss):
        print("Videos outside PDF lists (still plotted if any rows):")
        display(miss)

    agg = dict(
        mean_speed_mm_s=("speed_mm_s", lambda s: np.nanmean(s.values)),
        total_dist_mm=("total_dist_mm", "first"),
        n_rows=("frame", "count"),
    )
    if "w" in ok.columns:
        agg["mean_w_px"] = ("w", lambda s: np.nanmean(s.values))

    per_video = ok.groupby(["experiment_id", "detection_run_id", "condition"], as_index=False).agg(**agg)
    per_video["pooled"] = np.where(
        per_video["condition"].str.contains("dehydr", case=False, na=False),
        "dehydrated w1118",
        "w1118 on media",
    )
    display(per_video.sort_values(["experiment_id", "detection_run_id"]))

    # Helper run_dehydration_step0_style is defined inline above (no external module).
    df_w = ok.copy()
    df_w["category"] = df_w["condition"].astype(str)
    df_w["age_group"] = np.where(
        df_w["condition"].astype(str).str.contains("dehydr", case=False, na=False),
        "old",
        "young",
    )
    BASE_W = Path("pattern_distance_output") / "w1118_dehydration_step0"
    sum_w = run_dehydration_step0_style(
        df_w,
        video_cols,
        FEATURE_COLS,
        turn_stats_from_xy,
        BASE_W,
        hue_display_names=("Control (w1118)", "Dehydrated (w1118)"),
        title_prefix="w1118: control vs dehydrated (sd_09_08 + sd_09_09)",
        skip_figures=SKIP_PATTERN_FIGURES,
        only_cluster_naming_order_csv=ONLY_CLUSTER_NAMING_ORDER_CSV,
    )
    display(sum_w)
    print("Step 0–style outputs (8 PNGs each):", BASE_W.resolve())



### Sign-up list.pdf — which `detection_output_N.csv` is which

**SD 9-9-25** (folder `sd_09_09`, experiment id `sd_09_09`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1, 3, 5: w1118** · **Videos 2, 4, 6: dehydrated w1118**

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` | **w1118** (on media) |
| `detection_output_2.csv` | **dehydrated w1118** |
| `detection_output_3.csv` | **w1118** (on media) |
| `detection_output_4.csv` | **dehydrated w1118** |
| `detection_output_5.csv` | **w1118** (on media) |
| `detection_output_6.csv` | **dehydrated w1118** |

---

**SD 9-8-25** (folder `sd_09_08`, experiment id `sd_09_08`) — *Dehydration WIG Males and controls*  
PDF notes: **Videos 1–3: On media** · **Videos 4–7: Dehydrated**  
(We plot these as **w1118 / on-media** vs **w1118 dehydrated** to match the 9-9 cohort design.)

| CSV file | Condition (PDF wording) |
|----------|-------------------------|
| `detection_output_1.csv` … `detection_output_3.csv` | **On media** → pooled as **w1118 (on media)** |
| `detection_output_4.csv` … `detection_output_7.csv` | **Dehydrated** → pooled as **w1118 dehydrated** |

**N** is the video index (same as **`detection_run_id`**). Files may be named `detection_output_N.csv` or `detection_output_N_filtered.csv`.


,experiment_id,detection_run_id (video #),condition (from PDF)
0,sd_09_08,1,w1118 (on media)
1,sd_09_08,2,w1118 (on media)
2,sd_09_08,3,w1118 (on media)
3,sd_09_08,4,w1118 dehydrated
4,sd_09_08,5,w1118 dehydrated
5,sd_09_08,6,w1118 dehydrated
6,sd_09_08,7,w1118 dehydrated
7,sd_09_09,1,w1118 (on media)
8,sd_09_09,2,w1118 dehydrated
9,sd_09_09,3,w1118 (on media)


PDF → which CSV (same rules as `signup_w1118_condition`):


,experiment_id,N (video #),CSV filename,PDF → plot group
0,sd_09_09,1,detection_output_1.csv,w1118 (PDF: video 1)
1,sd_09_09,2,detection_output_2.csv,dehydrated w1118 (PDF: video 2)
2,sd_09_09,3,detection_output_3.csv,w1118 (PDF: video 3)
3,sd_09_09,4,detection_output_4.csv,dehydrated w1118 (PDF: video 4)
4,sd_09_09,5,detection_output_5.csv,w1118 (PDF: video 5)
5,sd_09_09,6,detection_output_6.csv,dehydrated w1118 (PDF: video 6)
6,sd_09_08,1,detection_output_1.csv,on media (PDF) → w1118 (on media)
7,sd_09_08,2,detection_output_2.csv,on media (PDF) → w1118 (on media)
8,sd_09_08,3,detection_output_3.csv,on media (PDF) → w1118 (on media)
9,sd_09_08,4,detection_output_4.csv,Dehydrated (PDF) → w1118 dehydrated


Reading individual CSVs under /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp2
Rows: 711,856  videos: 12


,experiment_id,detection_run_id,condition,mean_speed_mm_s,total_dist_mm,n_rows,mean_w_px,pooled
0,sd_09_08,2,w1118 (on media),6.760604,4621.796842,37831,43.400492,w1118 on media
1,sd_09_08,3,w1118 (on media),7.742756,3987.972313,29213,46.104029,w1118 on media
2,sd_09_08,4,w1118 dehydrated,10.104718,4151.282262,22002,51.948277,dehydrated w1118
3,sd_09_08,5,w1118 dehydrated,6.688828,7338.375020,56945,51.237317,dehydrated w1118
4,sd_09_08,6,w1118 dehydrated,7.045347,5524.941904,41015,48.436962,dehydrated w1118
5,sd_09_08,7,w1118 dehydrated,9.008417,9109.483415,57525,53.299435,dehydrated w1118
6,sd_09_09,1,w1118 (on media),7.549500,12362.376494,92094,49.164973,w1118 on media
7,sd_09_09,2,w1118 dehydrated,7.881726,13498.563076,95772,47.520361,dehydrated w1118
8,sd_09_09,3,w1118 (on media),8.281070,12883.835881,88807,50.878917,w1118 on media
9,sd_09_09,4,w1118 dehydrated,7.608609,11968.344947,89071,57.122374,dehydrated w1118


  w1118: control vs dehydrated (sd_09_08 + sd_09_09) 5mm: n_segments=9,109  ->  k=5  silhouette=0.3654
  w1118: control vs dehydrated (sd_09_08 + sd_09_09) 10mm: n_segments=9,277  ->  k=4  silhouette=0.3584
  w1118: control vs dehydrated (sd_09_08 + sd_09_09) 15mm: n_segments=6,352  ->  k=4  silhouette=0.3432
  w1118: control vs dehydrated (sd_09_08 + sd_09_09) 20mm: n_segments=4,848  ->  k=4  silhouette=0.3147
  w1118: control vs dehydrated (sd_09_08 + sd_09_09) 30mm: n_segments=3,279  ->  k=4  silhouette=0.2890
  w1118: control vs dehydrated (sd_09_08 + sd_09_09) 40mm: n_segments=2,481  ->  k=2  silhouette=0.8095
Summary: /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp2/pattern_distance_output/w1118_dehydration_step0/step0_pattern_summary.csv


,segment_length_mm,n_segments,n_patterns_k,n_distinct_labels,best_silhouette,labels
0,5,9109,5,5,0.365404,curved; meandering; straight; winding; zig-zag
1,10,9277,4,4,0.358389,curved; meandering; straight; zig-zag
2,15,6352,4,4,0.343162,curved; meandering; straight; zig-zag
3,20,4848,4,4,0.314694,curved; meandering; straight; zig-zag
4,30,3279,4,4,0.288995,curved; meandering; straight; zig-zag
5,40,2481,2,2,0.809521,straight; zig-zag


Step 0–style outputs (8 PNGs each): /Users/yashpatel/Documents/John-Tower-Lab/paper_trajectory/Exp2/pattern_distance_output/w1118_dehydration_step0
